# 04 · Monte Carlo loss distribution, dependence and uncertainty layers

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 20)
from offshore_risk import load_config
cfg = load_config()
from offshore_risk.bayesian.calibration import apply_posteriors
from offshore_risk.simulation import simulate, SOURCES
from offshore_risk.financial.metrics import risk_metrics, tail_allocation
from offshore_risk.visualization import plots
cfg = apply_posteriors(cfg)
b = simulate(cfg, n_years=50_000, dependence='correlated')
a = simulate(cfg, n_years=50_000, dependence='independent')
pd.DataFrame({'Model A': risk_metrics(a.total), 'Model B': risk_metrics(b.total)}).T[['eal', 'median', 'p90', 'p95', 'p99', 'es99']] / 1e6

,eal,median,p90,p95,p99,es99
Model A,22.105612,13.615642,40.579212,57.273455,174.320404,351.278110
Model B,22.708255,13.594962,42.553231,61.630732,181.192842,360.719927


In [2]:
plots.exceedance_curves({'Model A (independent)': a.total, 'Model B (correlated)': b.total})

<Figure size 880x495 with 1 Axes>

Correlated year-level drivers raise the middle-to-upper part of the distribution (P90–P95, where years with several moderate events accumulate) but hardly change the extreme tail, which is set by single major-accident events.

In [3]:
tail_allocation(b.loss_by_source, SOURCES, 0.99)

,source,mean_loss,share_of_eal,es99_contribution,share_of_es99
0,release,8.207708e+06,0.361442,2.416637e+08,0.669948
1,compressor,5.250455e+06,0.231213,8.506400e+06,0.023582
2,weather,3.247222e+06,0.142997,4.994037e+06,0.013845
3,spurious_trip,2.872451e+06,0.126494,3.645159e+06,0.010105
4,pipeline,2.152538e+06,0.094791,9.984790e+07,0.276802
5,power_loss,7.450422e+05,0.032809,1.008095e+06,0.002795
6,collision,2.328388e+05,0.010253,1.054667e+06,0.002924


In [4]:
b.component_frame().mean().to_frame('mean USD/yr')

,mean USD/yr
property_damage,9.693508e+05
equipment_repair,1.074602e+06
emergency_response,3.178208e+05
logistics,1.826306e+05
restart,1.536186e+06
business_interruption,1.862766e+07
environmental_scenario,0.000000e+00


## Epistemic vs aleatory
Each 'world' fixes the uncertain parameters; the spread within a world is year-to-year randomness, the spread between worlds is what we do not know.

In [5]:
from offshore_risk.simulation.nested import run_nested
nr = run_nested(cfg, n_outer=150, n_inner=800)
nr.metric_summary()

,metric,p05,median,mean,p95,ratio_p95_p05
0,eal,1.337786e+07,2.177200e+07,2.278780e+07,3.336438e+07,2.494000
1,p95,3.366578e+07,5.536820e+07,5.980684e+07,8.813012e+07,2.617795
2,p99,7.229955e+07,1.584680e+08,1.745685e+08,3.190924e+08,4.413477
3,es99,1.343157e+08,3.271717e+08,3.417911e+08,6.207308e+08,4.621432


In [6]:
plots.exceedance_band(nr, b.total)

<Figure size 880x495 with 1 Axes>